In [1]:
fixed_logistic = '''"""
Logistic Regression baseline model for the WorldCup Intelligence Platform.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import joblib
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    log_loss,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path(__file__).resolve().parents[3]

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "ml_matches.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "logistic_regression.joblib"

FEATURE_COLUMNS = [
    "home_elo", "away_elo", "elo_diff",
    "home_expected_elo", "away_expected_elo",
    "neutral", "home_advantage_applied",
    "tournament_k_factor",
    "home_matches_played", "away_matches_played", "matches_played_diff",
    "home_recent_points_per_match", "away_recent_points_per_match", "recent_points_diff",
    "home_recent_goals_for", "away_recent_goals_for", "recent_goals_for_diff",
    "home_recent_goals_against", "away_recent_goals_against", "recent_goals_against_diff",
    "home_recent_goal_difference", "away_recent_goal_difference", "recent_goal_difference_diff",
    "home_rest_days", "away_rest_days", "rest_days_diff",
    "head_to_head_matches",
    "home_head_to_head_points_per_match", "away_head_to_head_points_per_match",
    "head_to_head_points_diff",
]

TARGET_COLUMN = "target_code"
LABEL_MAP = {0: "Away Win", 1: "Draw", 2: "Home Win"}


@dataclass
class DatasetSplit:
    X_train: pd.DataFrame
    X_test: pd.DataFrame
    y_train: pd.Series
    y_test: pd.Series


@dataclass
class EvaluationResult:
    accuracy: float
    log_loss: float
    confusion_matrix: Any
    classification_report: dict[str, Any]


class LogisticMatchPredictor:

    def __init__(self, random_state: int = 42, test_size: float = 0.20) -> None:
        self.random_state = random_state
        self.test_size = test_size
        self.pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                multi_class="multinomial",
                solver="lbfgs",
                max_iter=1000,
                random_state=random_state,
            )),
        ])
        self.is_trained = False

    @staticmethod
    def load_dataset(dataset_path: Path | str = DATA_PATH) -> pd.DataFrame:
        df = pd.read_csv(dataset_path)
        df = df.sort_values("match_index").reset_index(drop=True)
        return df

    @staticmethod
    def prepare_features(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
        return df[FEATURE_COLUMNS].copy(), df[TARGET_COLUMN].copy()

    def chronological_split(self, df: pd.DataFrame) -> DatasetSplit:
        split_index = int(len(df) * (1 - self.test_size))
        train = df.iloc[:split_index]
        test = df.iloc[split_index:]
        return DatasetSplit(
            X_train=train[FEATURE_COLUMNS],
            X_test=test[FEATURE_COLUMNS],
            y_train=train[TARGET_COLUMN],
            y_test=test[TARGET_COLUMN],
        )

    def train(self, df: pd.DataFrame) -> None:
        dataset = self.chronological_split(df)
        self.pipeline.fit(dataset.X_train, dataset.y_train)
        self.is_trained = True

    def predict(self, features: pd.DataFrame) -> Any:
        if not self.is_trained:
            raise RuntimeError("Model has not been trained.")
        return self.pipeline.predict(features)

    def predict_proba(self, features: pd.DataFrame) -> Any:
        if not self.is_trained:
            raise RuntimeError("Model has not been trained.")
        return self.pipeline.predict_proba(features)

    def predict_match(
        self,
        home_team: str,
        away_team: str,
        home_elo: float,
        away_elo: float,
        neutral: bool = False,
        home_advantage: float = 75.0,
        tournament_k_factor: float = 20.0,
        home_matches_played: int = 50,
        away_matches_played: int = 50,
        home_recent_points_per_match: float = 1.5,
        away_recent_points_per_match: float = 1.5,
        home_recent_goals_for: float = 1.5,
        away_recent_goals_for: float = 1.5,
        home_recent_goals_against: float = 1.0,
        away_recent_goals_against: float = 1.0,
        home_recent_goal_difference: float = 0.5,
        away_recent_goal_difference: float = 0.5,
        home_rest_days: int = 7,
        away_rest_days: int = 7,
        head_to_head_matches: int = 0,
        home_h2h_points: float = 0.0,
        away_h2h_points: float = 0.0,
    ) -> dict[str, Any]:
        """Predict a single match given team context."""
        from worldcup_intelligence.elo import expected_scores

        home_expected, away_expected = expected_scores(
            home_rating=home_elo,
            away_rating=away_elo,
            neutral=neutral,
            home_advantage=home_advantage,
        )

        features = pd.DataFrame([{
            "home_elo": home_elo,
            "away_elo": away_elo,
            "elo_diff": home_elo - away_elo,
            "home_expected_elo": home_expected,
            "away_expected_elo": away_expected,
            "neutral": int(neutral),
            "home_advantage_applied": 0.0 if neutral else home_advantage,
            "tournament_k_factor": tournament_k_factor,
            "home_matches_played": home_matches_played,
            "away_matches_played": away_matches_played,
            "matches_played_diff": home_matches_played - away_matches_played,
            "home_recent_points_per_match": home_recent_points_per_match,
            "away_recent_points_per_match": away_recent_points_per_match,
            "recent_points_diff": home_recent_points_per_match - away_recent_points_per_match,
            "home_recent_goals_for": home_recent_goals_for,
            "away_recent_goals_for": away_recent_goals_for,
            "recent_goals_for_diff": home_recent_goals_for - away_recent_goals_for,
            "home_recent_goals_against": home_recent_goals_against,
            "away_recent_goals_against": away_recent_goals_against,
            "recent_goals_against_diff": home_recent_goals_against - away_recent_goals_against,
            "home_recent_goal_difference": home_recent_goal_difference,
            "away_recent_goal_difference": away_recent_goal_difference,
            "recent_goal_difference_diff": home_recent_goal_difference - away_recent_goal_difference,
            "home_rest_days": home_rest_days,
            "away_rest_days": away_rest_days,
            "rest_days_diff": home_rest_days - away_rest_days,
            "head_to_head_matches": head_to_head_matches,
            "home_head_to_head_points_per_match": home_h2h_points,
            "away_head_to_head_points_per_match": away_h2h_points,
            "head_to_head_points_diff": home_h2h_points - away_h2h_points,
        }])

        proba = self.predict_proba(features)[0]

        return {
            "home_team": home_team,
            "away_team": away_team,
            "home_win_probability": round(float(proba[2]), 4),
            "draw_probability": round(float(proba[1]), 4),
            "away_win_probability": round(float(proba[0]), 4),
            "predicted_winner": (
                home_team if proba[2] > proba[0] and proba[2] > proba[1]
                else away_team if proba[0] > proba[2] and proba[0] > proba[1]
                else "Draw"
            ),
            "confidence": round(float(max(proba)), 4),
            "home_elo": home_elo,
            "away_elo": away_elo,
            "model": "LogisticRegression",
        }

    def evaluate(self, df: pd.DataFrame) -> EvaluationResult:
        if not self.is_trained:
            raise RuntimeError("Model has not been trained.")
        dataset = self.chronological_split(df)
        predictions = self.pipeline.predict(dataset.X_test)
        probabilities = self.pipeline.predict_proba(dataset.X_test)
        return EvaluationResult(
            accuracy=accuracy_score(dataset.y_test, predictions),
            log_loss=log_loss(dataset.y_test, probabilities),
            confusion_matrix=confusion_matrix(dataset.y_test, predictions),
            classification_report=classification_report(
                dataset.y_test, predictions, output_dict=True
            ),
        )

    def train_and_evaluate(self, dataset_path: Path | str = DATA_PATH) -> EvaluationResult:
        df = self.load_dataset(dataset_path)
        self.train(df)
        return self.evaluate(df)

    def save_model(self, output_path: Path | str = MODEL_PATH) -> None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(self.pipeline, output_path)

    def load_model(self, model_path: Path | str = MODEL_PATH) -> None:
        self.pipeline = joblib.load(model_path)
        self.is_trained = True

    @property
    def coefficients(self) -> pd.DataFrame:
        if not self.is_trained:
            raise RuntimeError("Model has not been trained.")
        classifier = self.pipeline.named_steps["classifier"]
        return pd.DataFrame(classifier.coef_, columns=FEATURE_COLUMNS)

    @property
    def intercept(self) -> Any:
        if not self.is_trained:
            raise RuntimeError("Model has not been trained.")
        return self.pipeline.named_steps["classifier"].intercept_
'''

output_path = r"E:\Python\worldcup-intelligence-platform\src\worldcup_intelligence\models\logistic.py"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(fixed_logistic)

print("✅ logistic.py fixed and written successfully.")

✅ logistic.py fixed and written successfully.


In [2]:
import sys
sys.path.insert(0, r"E:\Python\worldcup-intelligence-platform\src")
sys.path.insert(0, r"E:\Python\worldcup-intelligence-platform")

from worldcup_intelligence.models.logistic import LogisticMatchPredictor

predictor = LogisticMatchPredictor()
result = predictor.train_and_evaluate()

print(f"✅ Accuracy:  {result.accuracy:.4f}")
print(f"✅ Log Loss:  {result.log_loss:.4f}")
print(f"\nConfusion Matrix:\n{result.confusion_matrix}")

✅ Accuracy:  0.6022
✅ Log Loss:  0.8715

Confusion Matrix:
[[ 951   48  502]
 [ 415   40  706]
 [ 311   39 2069]]


In [3]:
predictor.save_model()
print("✅ Model saved to models/logistic_regression.joblib")

✅ Model saved to models/logistic_regression.joblib


In [4]:
from pathlib import Path

ROOT = Path(r"E:\Python\worldcup-intelligence-platform")
app_path = ROOT / "dashboard" / "app.py"

lines = []
lines.append('import streamlit as st')
lines.append('import pandas as pd')
lines.append('import sys')
lines.append('from pathlib import Path')
lines.append('import matplotlib.pyplot as plt')
lines.append('import numpy as np')
lines.append('')
lines.append('ROOT = Path(__file__).resolve().parents[1]')
lines.append('sys.path.insert(0, str(ROOT / "src"))')
lines.append('sys.path.insert(0, str(ROOT))')
lines.append('')
lines.append('from worldcup_intelligence.models.logistic import LogisticMatchPredictor')
lines.append('from worldcup_intelligence.elo import EloRatingEngine')
lines.append('')
lines.append('st.set_page_config(page_title="WorldCup Intelligence Platform", page_icon="⚽", layout="wide")')
lines.append('')
lines.append('st.markdown("""')
lines.append('<style>')
lines.append('.winner-badge {')
lines.append('    background: linear-gradient(135deg, #ffd700, #ffaa00);')
lines.append('    color: black; border-radius: 20px; padding: 8px 24px;')
lines.append('    font-weight: bold; font-size: 1.1em; display: inline-block;')
lines.append('}')
lines.append('</style>')
lines.append('""", unsafe_allow_html=True)')
lines.append('')
lines.append('@st.cache_resource')
lines.append('def load_model():')
lines.append('    predictor = LogisticMatchPredictor()')
lines.append('    predictor.load_model(ROOT / "models" / "logistic_regression.joblib")')
lines.append('    return predictor')
lines.append('')
lines.append('@st.cache_data')
lines.append('def load_elo_ratings():')
lines.append('    df = pd.read_csv(ROOT / "data" / "processed" / "ml_matches.csv")')
lines.append('    df = df.sort_values("match_index").reset_index(drop=True)')
lines.append('    engine = EloRatingEngine()')
lines.append('    for _, row in df.iterrows():')
lines.append('        engine.process_match(')
lines.append('            home_team=row["home_team"], away_team=row["away_team"],')
lines.append('            home_score=int(row["home_score"]), away_score=int(row["away_score"]),')
lines.append('            tournament=row.get("tournament"), neutral=row.get("neutral", False),')
lines.append('            date=str(row["date"]) if pd.notna(row.get("date")) else None,')
lines.append('        )')
lines.append('    return engine')
lines.append('')
lines.append('@st.cache_data')
lines.append('def get_team_list():')
lines.append('    df = pd.read_csv(ROOT / "data" / "processed" / "ml_matches.csv")')
lines.append('    return sorted(set(df["home_team"].tolist() + df["away_team"].tolist()))')
lines.append('')
lines.append('@st.cache_data')
lines.append('def get_team_history(team):')
lines.append('    df = pd.read_csv(ROOT / "data" / "processed" / "ml_matches.csv")')
lines.append('    df = df.sort_values("match_index").reset_index(drop=True)')
lines.append('    engine = EloRatingEngine()')
lines.append('    history = []')
lines.append('    for _, row in df.iterrows():')
lines.append('        ht, at = row["home_team"], row["away_team"]')
lines.append('        if ht == team or at == team:')
lines.append('            history.append({"match_index": row["match_index"], "elo": engine.get_team_rating(team)})')
lines.append('        engine.process_match(')
lines.append('            home_team=ht, away_team=at,')
lines.append('            home_score=int(row["home_score"]), away_score=int(row["away_score"]),')
lines.append('            tournament=row.get("tournament"), neutral=row.get("neutral", False),')
lines.append('            date=str(row["date"]) if pd.notna(row.get("date")) else None,')
lines.append('        )')
lines.append('    return pd.DataFrame(history)')
lines.append('')
lines.append('def make_prediction(home, away, neutral, k):')
lines.append('    engine = load_elo_ratings()')
lines.append('    return predictor.predict_match(')
lines.append('        home_team=home, away_team=away,')
lines.append('        home_elo=engine.get_team_rating(home),')
lines.append('        away_elo=engine.get_team_rating(away),')
lines.append('        neutral=neutral, tournament_k_factor=k,')
lines.append('    )')
lines.append('')
lines.append('def prob_bar(h, d, a, home, away):')
lines.append('    fig, ax = plt.subplots(figsize=(8, 0.8))')
lines.append('    fig.patch.set_facecolor("#0e1117")')
lines.append('    ax.set_facecolor("#0e1117")')
lines.append('    left = 0')
lines.append('    for v, c, lbl in zip([h, d, a], ["#00c853", "#ffd600", "#ff1744"], [home, "Draw", away]):')
lines.append('        ax.barh(0, v, left=left, color=c, height=0.6)')
lines.append('        if v > 0.07:')
lines.append('            ax.text(left + v/2, 0, f"{lbl}\\n{v*100:.1f}%", ha="center", va="center",')
lines.append('                    color="black", fontsize=9, fontweight="bold")')
lines.append('        left += v')
lines.append('    ax.set_xlim(0, 1)')
lines.append('    ax.axis("off")')
lines.append('    st.pyplot(fig, use_container_width=True)')
lines.append('    plt.close()')
lines.append('')
lines.append('predictor = load_model()')
lines.append('teams = get_team_list()')
lines.append('')
lines.append('# ── Header ───────────────────────────────────────────────────────')
lines.append('st.title("⚽ WorldCup Intelligence Platform")')
lines.append('st.markdown("*ML-powered match predictions · FIFA World Cup 2026*")')
lines.append('')
lines.append('# Sidebar')
lines.append('st.sidebar.title("⚽ WorldCup Intelligence")')
lines.append('st.sidebar.markdown("---")')
lines.append('st.sidebar.markdown("### 🏆 FIFA World Cup 2026")')
lines.append('st.sidebar.markdown("**Semi-Finals**")')
lines.append('st.sidebar.markdown("🇫🇷 France vs Spain 🇪🇸  \\n*July 14 · Dallas*")')
lines.append('st.sidebar.markdown("🏴󠁧󠁢󠁥󠁮󠁧󠁿 England vs Argentina 🇦🇷  \\n*July 15 · Atlanta*")')
lines.append('st.sidebar.markdown("**Final**")')
lines.append('st.sidebar.markdown("🏟️ July 19 · MetLife Stadium, NJ")')
lines.append('st.sidebar.markdown("---")')
lines.append('st.sidebar.markdown("**Model:** Logistic Regression  \\n**Accuracy:** 60.2%  \\n**Dataset:** 25,403 matches")')
lines.append('st.sidebar.markdown("---")')
lines.append('st.sidebar.markdown("**Made by Bhavya Sharma**  \\n[GitHub](https://github.com) · [LinkedIn](https://linkedin.com)")')
lines.append('')
lines.append('st.markdown("---")')
lines.append('')
lines.append('# ── Semi-final predictions ───────────────────────────────────────')
lines.append('st.markdown("## 🔥 Semi-Final Predictions")')
lines.append('col1, col2 = st.columns(2)')
lines.append('')
lines.append('SEMIS = [')
lines.append('    {"home": "France", "away": "Spain", "label": "SF1 · July 14 · Dallas", "neutral": True, "k": 60},')
lines.append('    {"home": "England", "away": "Argentina", "label": "SF2 · July 15 · Atlanta", "neutral": True, "k": 60},')
lines.append(']')
lines.append('')
lines.append('for semi, col in zip(SEMIS, [col1, col2]):')
lines.append('    pred = make_prediction(semi["home"], semi["away"], semi["neutral"], semi["k"])')
lines.append('    h = pred["home_win_probability"]')
lines.append('    d = pred["draw_probability"]')
lines.append('    a = pred["away_win_probability"]')
lines.append('    home = pred["home_team"]')
lines.append('    away = pred["away_team"]')
lines.append('    with col:')
lines.append('        st.markdown(f"**{semi[\'label\']}**")')
lines.append('        m1, m2, m3 = st.columns(3)')
lines.append('        m1.metric(home, f"{h*100:.1f}%", f"Elo: {pred[\'home_elo\']:.0f}")')
lines.append('        m2.metric("Draw", f"{d*100:.1f}%")')
lines.append('        m3.metric(away, f"{a*100:.1f}%", f"Elo: {pred[\'away_elo\']:.0f}")')
lines.append('        prob_bar(h, d, a, home, away)')
lines.append('        winner = pred["predicted_winner"]')
lines.append('        conf = pred["confidence"]')
lines.append('        st.markdown(f"🏆 **Predicted winner: {winner}** · Confidence: {conf*100:.1f}%")')
lines.append('        st.markdown("---")')
lines.append('')
lines.append('# ── Custom predictor ─────────────────────────────────────────────')
lines.append('st.markdown("## 🔮 Predict Any Match")')
lines.append('c1, c2, c3 = st.columns([2, 2, 1])')
lines.append('with c1:')
lines.append('    home_team = st.selectbox("🏠 Home Team", teams, index=teams.index("France") if "France" in teams else 0)')
lines.append('with c2:')
lines.append('    away_team = st.selectbox("✈️ Away Team", teams, index=teams.index("Spain") if "Spain" in teams else 1)')
lines.append('with c3:')
lines.append('    neutral = st.checkbox("Neutral Venue", value=True)')
lines.append('')
lines.append('tournament = st.selectbox("Tournament", ["FIFA World Cup", "FIFA World Cup qualification",')
lines.append('    "UEFA Euro", "Copa America", "Friendly", "UEFA Nations League", "African Cup of Nations"])')
lines.append('k_map = {"FIFA World Cup": 60, "FIFA World Cup qualification": 40, "UEFA Euro": 50,')
lines.append('         "Copa America": 50, "Friendly": 20, "UEFA Nations League": 35, "African Cup of Nations": 50}')
lines.append('')
lines.append('if st.button("⚡ Predict Match", type="primary", use_container_width=True):')
lines.append('    if home_team == away_team:')
lines.append('        st.error("Please select two different teams.")')
lines.append('    else:')
lines.append('        pred = make_prediction(home_team, away_team, neutral, k_map.get(tournament, 20))')
lines.append('        h, d, a = pred["home_win_probability"], pred["draw_probability"], pred["away_win_probability"]')
lines.append('        st.markdown("### Result")')
lines.append('        r1, r2, r3, r4 = st.columns(4)')
lines.append('        r1.metric(f"🏠 {home_team}", f"{h*100:.1f}%")')
lines.append('        r2.metric("🤝 Draw", f"{d*100:.1f}%")')
lines.append('        r3.metric(f"✈️ {away_team}", f"{a*100:.1f}%")')
lines.append('        r4.metric("🏆 Winner", pred["predicted_winner"])')
lines.append('        prob_bar(h, d, a, home_team, away_team)')
lines.append('        st.info(f"Elo Ratings — {home_team}: {pred[\'home_elo\']:.0f} · {away_team}: {pred[\'away_elo\']:.0f}")')
lines.append('')
lines.append('# ── Rankings ─────────────────────────────────────────────────────')
lines.append('st.markdown("## 📊 Current Elo Rankings (Top 30)")')
lines.append('engine = load_elo_ratings()')
lines.append('rankings = engine.get_rankings(limit=30)')
lines.append('rdf = pd.DataFrame(rankings)')
lines.append('rdf.columns = ["Rank", "Team", "Elo Rating"]')
lines.append('rdf["Elo Rating"] = rdf["Elo Rating"].round(1)')
lines.append('highlight = {"France", "Spain", "England", "Argentina"}')
lines.append('def hl(row):')
lines.append('    if row["Team"] in highlight:')
lines.append('        return ["background-color: #2d3250; font-weight: bold"] * len(row)')
lines.append('    return [""] * len(row)')
lines.append('st.dataframe(rdf.style.apply(hl, axis=1), use_container_width=True, hide_index=True, height=600)')
lines.append('')
lines.append('# ── Rating history ───────────────────────────────────────────────')
lines.append('st.markdown("## 📈 Team Rating History")')
lines.append('selected = st.multiselect("Select teams", teams, default=["France", "Spain", "England", "Argentina"])')
lines.append('if selected:')
lines.append('    fig, ax = plt.subplots(figsize=(12, 5))')
lines.append('    fig.patch.set_facecolor("#0e1117")')
lines.append('    ax.set_facecolor("#0e1117")')
lines.append('    colors = ["#00c853", "#ff1744", "#2196f3", "#ffd600", "#9c27b0", "#ff6090"]')
lines.append('    for i, team in enumerate(selected):')
lines.append('        hist = get_team_history(team)')
lines.append('        if not hist.empty:')
lines.append('            ax.plot(hist["match_index"], hist["elo"], label=team,')
lines.append('                    color=colors[i % len(colors)], linewidth=2)')
lines.append('    ax.set_xlabel("Match Index", color="#aaaaaa")')
lines.append('    ax.set_ylabel("Elo Rating", color="#aaaaaa")')
lines.append('    ax.set_title("Elo Rating History", color="#ffffff")')
lines.append('    ax.tick_params(colors="#aaaaaa")')
lines.append('    ax.spines["bottom"].set_color("#3d4570")')
lines.append('    ax.spines["left"].set_color("#3d4570")')
lines.append('    ax.spines["top"].set_visible(False)')
lines.append('    ax.spines["right"].set_visible(False)')
lines.append('    ax.legend(facecolor="#1e2130", labelcolor="#ffffff")')
lines.append('    ax.grid(alpha=0.15, color="#3d4570")')
lines.append('    st.pyplot(fig, use_container_width=True)')
lines.append('    plt.close()')
lines.append('')
lines.append('# ── Footer ───────────────────────────────────────────────────────')
lines.append('st.markdown("---")')
lines.append('st.markdown("<center><sub>WorldCup Intelligence Platform · Built with Python, Scikit-learn & Streamlit · Trained on 25,403 international matches (2000–2026)</sub></center>", unsafe_allow_html=True)')

app_path.write_text('\n'.join(lines), encoding='utf-8')
print("✅ dashboard/app.py written cleanly")

✅ dashboard/app.py written cleanly


In [5]:
from pathlib import Path

path = Path(r"E:\Python\worldcup-intelligence-platform\src\worldcup_intelligence\models\logistic.py")
content = path.read_text(encoding="utf-8")
content = content.replace(
    '''("classifier", LogisticRegression(
                multi_class="multinomial",
                solver="lbfgs",
                max_iter=1000,
                random_state=random_state,
            )),''',
    '''("classifier", LogisticRegression(
                solver="lbfgs",
                max_iter=1000,
                random_state=random_state,
            )),'''
)
path.write_text(content, encoding="utf-8")
print("✅ Fixed")

✅ Fixed


In [6]:
import sys
sys.path.insert(0, r"E:\Python\worldcup-intelligence-platform\src")
sys.path.insert(0, r"E:\Python\worldcup-intelligence-platform")

from worldcup_intelligence.models.logistic import LogisticMatchPredictor

predictor = LogisticMatchPredictor()
result = predictor.train_and_evaluate()
predictor.save_model()

print(f"✅ Retrained — Accuracy: {result.accuracy:.4f}, Log Loss: {result.log_loss:.4f}")
print("✅ Model saved")

✅ Retrained — Accuracy: 0.6022, Log Loss: 0.8715
✅ Model saved
